# Syria Crop Suitability PoC - Training Notebook

This notebook trains first-screening suitability models for olive and Damask rose.


## 1. Setup


In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(ROOT / 'src'))
ROOT


WindowsPath('c:/Users/atsumilab/Desktop/out/vega/crop_suitability_poc')

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from crop_profiles import load_crop_profiles, load_project_config
from suitability import add_all_crop_rule_scores, add_best_crop_columns
from utils import load_feature_columns, read_csv_safely, validate_columns, coerce_numeric, suitability_class


## 2. Load data
Run `src/make_demo_data.py` first if you do not yet have a GEE export.


In [3]:
input_csv = ROOT / 'data/raw/syria_crop_features_2025.csv'
if not input_csv.exists():
    print('Demo data not found. Run: python src/make_demo_data.py --output data/raw/demo_syria_crop_features.csv --n 5000')
else:
    df = pd.read_csv(input_csv)
    df.head()


## 3. Compute rule scores


In [4]:
profiles = load_crop_profiles(ROOT / 'config/crop_profiles.yaml')
scored = add_all_crop_rule_scores(df, profiles)
scored[[c for c in scored.columns if 'rule_score' in c]].describe()


,olive_rule_score,damask_rose_rule_score
count,37871.000000,37871.000000
mean,61.472323,48.270296
std,10.165365,11.309923
min,33.130000,18.070000
25%,55.130000,41.560000
50%,61.990000,46.760000
75%,65.460000,52.365000
max,88.000000,88.000000


## 4. Train models
The CLI script is the recommended training path. In the notebook, use this shell command:


In [ ]:
!python ../src/train.py --input ../data/raw/syria_crop_features_2025.csv --output ../data/processed/crop_suitability_predictions_notebook.csv     --crop-profiles ../config/crop_profiles.yaml  --project-config ../config/project.yaml    --feature-config config/feature_columns.json

Input rows used: 37,871
Wrote predictions: ..\data\processed\crop_suitability_predictions_notebook.csv
Metrics:
  olive: MAE=0.34, R2=0.995
  damask_rose: MAE=0.52, R2=0.992


training after remving lon lat

In [5]:
!python ../src/train.py --input ../data/raw/syria_crop_features_2025.csv --output ../data/processed/crop_suitability_predictions_no_lonlat.csv     --crop-profiles ../config/crop_profiles.yaml  --project-config ../config/project.yaml    --feature-config config/feature_columns.json

Model feature columns:
  - BLUE
  - GREEN
  - RED
  - NIR
  - RE1
  - RE2
  - RE3
  - SWIR1
  - SWIR2
  - NDVI
  - EVI
  - SAVI
  - NDWI
  - NDMI
  - NDBI
  - BSI
  - soil_ph
  - soil_org_carbon
  - slope_deg
  - elevation_m
  - mean_temp_c
  - annual_rain_mm
Input rows used: 37,871
Wrote predictions: ..\data\processed\crop_suitability_predictions_no_lonlat.csv
Metrics:
  olive: MAE=0.34, R2=0.995
  damask_rose: MAE=0.52, R2=0.992


## 5. Inspect predictions


In [10]:
pred = pd.read_csv(ROOT / 'data/processed/crop_suitability_predictions_notebook.csv')
pred[['olive_model_score','damask_rose_model_score','best_crop','best_crop_score']].head()


,olive_model_score,damask_rose_model_score,best_crop,best_crop_score
0,56.47,53.93,olive,56.47
1,60.31,45.81,olive,60.31
2,60.13,42.68,olive,60.13
3,56.58,52.95,olive,56.58
4,55.75,46.94,olive,55.75


In [6]:
pred = pd.read_csv(ROOT / 'data/processed/crop_suitability_predictions_no_lonlat.csv')
pred[['olive_model_score','damask_rose_model_score','best_crop','best_crop_score']].head()


,olive_model_score,damask_rose_model_score,best_crop,best_crop_score
0,56.47,53.93,olive,56.47
1,60.31,45.81,olive,60.31
2,60.13,42.68,olive,60.13
3,56.58,52.95,olive,56.58
4,55.75,46.94,olive,55.75


In [ ]:
pred['olive_model_score'].hist(bins=30)
plt.title('Olive suitability score')
plt.xlabel('Score')
plt.ylabel('Count')
plt.show()


In [ ]:
pred['damask_rose_model_score'].hist(bins=30)
plt.title('Damask rose suitability score')
plt.xlabel('Score')
plt.ylabel('Count')
plt.show()


## 6. Run dashboard
From project root:

```bash
streamlit run app/streamlit_app.py
```
